# Itchy — MegaByte-style Causal Patching

Global transformer processes patches for efficiency.
Small autoregressive local decoder predicts bytes within each patch.
Shift-by-1, but the local decoder is CAUSAL — byte k only sees bytes 0..k-1.
No leakage.

**Runtime > H100 or A100 > Run all**

In [ ]:
# ============================================================
# SETUP
# ============================================================
!pip install -q torch numpy huggingface-hub sentencepiece

import os, shutil, math, time, numpy as np
from pathlib import Path
from huggingface_hub import hf_hub_download
import sentencepiece as spm
import torch
import torch.nn.functional as F
from torch import nn

assert torch.cuda.is_available(), 'No GPU!'
device = torch.device('cuda')
gpu = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu} ({mem:.0f}GB)')

In [ ]:
# ============================================================
# DATA — 2 shards
# ============================================================
os.makedirs('data/bytes', exist_ok=True)

def dl(fn, sub, dst):
    if Path(dst).exists(): return
    print(f'  Downloading {fn}...')
    c = str(Path(hf_hub_download(repo_id='willdepueoai/parameter-golf',
        filename=fn, subfolder=sub, repo_type='dataset')).resolve())
    try: os.link(c, dst)
    except OSError: shutil.copy2(c, dst)

dl('fineweb_train_000000.bin', 'datasets/datasets/fineweb10B_sp1024', 'data/train0.bin')
dl('fineweb_train_000001.bin', 'datasets/datasets/fineweb10B_sp1024', 'data/train1.bin')
dl('fineweb_val_000000.bin', 'datasets/datasets/fineweb10B_sp1024', 'data/val.bin')
dl('fineweb_1024_bpe.model', 'datasets/tokenizers', 'data/tok.model')

def load_shard(p):
    h = np.fromfile(p, dtype='<i4', count=256)
    return np.fromfile(p, dtype='<u2', count=int(h[2]), offset=256*4)

def write_shard(p, d):
    h = np.zeros(256, dtype='<i4')
    h[0] = 20240520; h[1] = 1; h[2] = len(d)
    with open(p, 'wb') as f:
        f.write(h.tobytes())
        f.write(d.astype('<u2').tobytes())

sp = spm.SentencePieceProcessor(model_file='data/tok.model')
for src, dst in [('data/train0.bin','data/bytes/train0.bin'),
                 ('data/train1.bin','data/bytes/train1.bin'),
                 ('data/val.bin','data/bytes/val.bin')]:
    if Path(dst).exists(): continue
    print(f'  Converting {src}...')
    tok = load_shard(src)
    bs = [np.frombuffer(sp.decode(tok[j:j+100000].tolist()).encode('utf-8'), dtype=np.uint8)
          for j in range(0, len(tok), 100000)]
    write_shard(dst, np.concatenate(bs))

def load_bytes(p):
    h = np.fromfile(p, dtype='<i4', count=256)
    return np.fromfile(p, dtype='<u2', count=int(h[2]), offset=256*4).astype(np.int64)

train_data = np.concatenate([load_bytes('data/bytes/train0.bin'), load_bytes('data/bytes/train1.bin')])
val_data = load_bytes('data/bytes/val.bin')
print(f'Train: {len(train_data):,} bytes | Val: {len(val_data):,} bytes')

In [ ]:
# ============================================================
# MODEL — MegaByte-style: global patches + causal local decoder
# ============================================================
#
# How it works:
# 1. Input bytes are grouped into non-overlapping patches of P
# 2. Each patch is embedded (concat byte embeddings + project)
# 3. Global transformer processes patches with causal attention
# 4. For each patch, a small LOCAL decoder autoregressively
#    predicts the P target bytes, conditioned on:
#    - The global patch representation (what context says comes next)
#    - The previous target bytes (autoregressive within patch)
#
# Target: shift by 1 (standard next-byte)
# x = data[:-1], y = data[1:]
#
# Input patch i = x[i*P : (i+1)*P] = data[i*P : (i+1)*P]
# Target patch i = y[i*P : (i+1)*P] = data[i*P+1 : (i+1)*P+1]
#
# The local decoder for patch i receives:
# - global_repr[i]: from global transformer (context from patches 0..i)
# - For predicting target byte j in patch i:
#   Only sees target bytes 0..j-1 (causal). Does NOT see input bytes
#   from the same patch that overlap with targets.
#
# This is honest because the local decoder is autoregressive
# and only conditions on previously predicted bytes + global context.

class Rotary(nn.Module):
    def __init__(self, dim, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        self.register_buffer('inv_freq', inv_freq, persistent=False)
        self._cache = None
    def forward(self, seq_len, device, dtype):
        if self._cache is None or self._cache[0] != seq_len:
            t = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype)
            freqs = torch.outer(t, self.inv_freq.to(device))
            self._cache = (seq_len, freqs.cos()[None,None,:,:], freqs.sin()[None,None,:,:])
        return self._cache[1].to(dtype), self._cache[2].to(dtype)

def apply_rope(x, cos, sin):
    h = x.size(-1) // 2
    x1, x2 = x[..., :h], x[..., h:]
    return torch.cat((x1*cos + x2*sin, x1*(-sin) + x2*cos), dim=-1)

class Attn(nn.Module):
    def __init__(self, dim, nh, nkv, qk_gain):
        super().__init__()
        self.nh, self.nkv, self.hd = nh, nkv, dim // nh
        kv = nkv * self.hd
        self.cq = nn.Linear(dim, dim, bias=False)
        self.ck = nn.Linear(dim, kv, bias=False)
        self.cv = nn.Linear(dim, kv, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)
        nn.init.zeros_(self.proj.weight)
        self.qg = nn.Parameter(torch.full((nh,), qk_gain))
        self.rot = Rotary(self.hd)
    def forward(self, x):
        B, S, D = x.shape
        q = self.cq(x).reshape(B,S,self.nh,self.hd).transpose(1,2)
        k = self.ck(x).reshape(B,S,self.nkv,self.hd).transpose(1,2)
        v = self.cv(x).reshape(B,S,self.nkv,self.hd).transpose(1,2)
        q, k = F.rms_norm(q,(self.hd,)), F.rms_norm(k,(self.hd,))
        cos, sin = self.rot(S, x.device, q.dtype)
        q, k = apply_rope(q,cos,sin), apply_rope(k,cos,sin)
        q = q * self.qg.to(q.dtype)[None,:,None,None]
        y = F.scaled_dot_product_attention(q,k,v, is_causal=True, enable_gqa=(self.nkv!=self.nh))
        return self.proj(y.transpose(1,2).contiguous().reshape(B,S,D))

class MLP(nn.Module):
    def __init__(self, dim, mult):
        super().__init__()
        self.fc = nn.Linear(dim, dim*mult, bias=False)
        self.proj = nn.Linear(dim*mult, dim, bias=False)
        nn.init.zeros_(self.proj.weight)
    def forward(self, x):
        return self.proj(F.leaky_relu(self.fc(x), 0.5).square())

class GlobalBlock(nn.Module):
    def __init__(self, dim, nh, nkv, mult, qk_gain):
        super().__init__()
        self.attn = Attn(dim, nh, nkv, qk_gain)
        self.mlp = MLP(dim, mult)
        self.as_ = nn.Parameter(torch.ones(dim))
        self.ms = nn.Parameter(torch.ones(dim))
        self.rm = nn.Parameter(torch.stack((torch.ones(dim), torch.zeros(dim))))
    def forward(self, x, x0):
        m = self.rm.to(x.dtype)
        x = m[0][None,None,:]*x + m[1][None,None,:]*x0
        x = x + self.as_.to(x.dtype)[None,None,:] * self.attn(F.rms_norm(x,(x.size(-1),)))
        x = x + self.ms.to(x.dtype)[None,None,:] * self.mlp(F.rms_norm(x,(x.size(-1),)))
        return x


class LocalDecoder(nn.Module):
    """Small causal transformer that predicts bytes within a patch.
    
    Input: global patch representation + previous target bytes (teacher forced).
    Output: logits for each byte position.
    
    The global repr is prepended as the first token. Then target bytes
    0..P-2 are fed in (shifted right). Causal attention ensures byte k
    only sees the global repr + bytes 0..k-1. Predictions for all P
    positions come out.
    """
    def __init__(self, global_dim, local_dim, patch_size, num_layers=2, nh=4):
        super().__init__()
        self.ps = patch_size
        self.local_dim = local_dim
        # Project global repr to local dim
        self.global_proj = nn.Linear(global_dim, local_dim, bias=False)
        # Byte embedding for target bytes (teacher forcing input)
        self.byte_embed = nn.Embedding(260, local_dim)
        # Learned position embeddings for positions within patch
        self.pos_embed = nn.Parameter(torch.randn(patch_size + 1, local_dim) * 0.02)  # +1 for global token
        # Local transformer layers
        self.layers = nn.ModuleList()
        for _ in range(num_layers):
            self.layers.append(nn.ModuleDict({
                'q': nn.Linear(local_dim, local_dim, bias=False),
                'k': nn.Linear(local_dim, local_dim, bias=False),
                'v': nn.Linear(local_dim, local_dim, bias=False),
                'proj': nn.Linear(local_dim, local_dim, bias=False),
                'mlp_fc': nn.Linear(local_dim, local_dim * 2, bias=False),
                'mlp_proj': nn.Linear(local_dim * 2, local_dim, bias=False),
            }))
            nn.init.zeros_(self.layers[-1]['proj'].weight)
            nn.init.zeros_(self.layers[-1]['mlp_proj'].weight)
        self.nh = nh
        self.hd = local_dim // nh
        # Output head
        self.head = nn.Linear(local_dim, 260, bias=False)

    def forward(self, global_repr, target_bytes):
        """
        global_repr: (B*NP, global_dim) — one repr per patch
        target_bytes: (B*NP, P) — target bytes for teacher forcing
        Returns: (B*NP, P, 260) — logits for each position
        """
        BNP = global_repr.shape[0]
        P = self.ps

        # Build input sequence: [global_token, target[0], target[1], ..., target[P-2]]
        # This has P tokens. Causal attention.
        # Output at position 0 predicts target[0] (from global context only)
        # Output at position k predicts target[k] (from global + target[0..k-1])
        
        global_tok = self.global_proj(global_repr).unsqueeze(1)  # (BNP, 1, local_dim)
        
        # Shift right: input is target[0..P-2], predict target[0..P-1]
        shifted = target_bytes[:, :-1]  # (BNP, P-1)
        byte_tok = self.byte_embed(shifted)  # (BNP, P-1, local_dim)
        
        # Concatenate: [global, byte0, byte1, ..., byte_{P-2}] = P tokens
        x = torch.cat([global_tok, byte_tok], dim=1)  # (BNP, P, local_dim)
        x = x + self.pos_embed[:P][None, :, :]  # add position embeddings
        
        # Run local transformer with causal attention
        for layer in self.layers:
            h = F.rms_norm(x, (self.local_dim,))
            q = layer['q'](h).reshape(BNP, P, self.nh, self.hd).transpose(1,2)
            k = layer['k'](h).reshape(BNP, P, self.nh, self.hd).transpose(1,2)
            v = layer['v'](h).reshape(BNP, P, self.nh, self.hd).transpose(1,2)
            attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
            x = x + layer['proj'](attn_out.transpose(1,2).reshape(BNP, P, self.local_dim))
            h = F.rms_norm(x, (self.local_dim,))
            x = x + layer['mlp_proj'](F.relu(layer['mlp_fc'](h)))
        
        return self.head(F.rms_norm(x, (self.local_dim,)))  # (BNP, P, 260)


class ItchyMegaByte(nn.Module):
    def __init__(self, global_dim, local_dim, global_layers, local_layers,
                 nh, nkv, mlp_mult, patch_size, local_nh=4, softcap=30.0):
        super().__init__()
        self.gdim = global_dim
        self.ps = patch_size
        self.sc = softcap
        
        # Patch embedding: embed each byte, concat, project
        self.byte_embed = nn.Embedding(260, global_dim)
        self.patch_proj = nn.Linear(global_dim * patch_size, global_dim, bias=False)
        
        # Global transformer
        ne = global_layers // 2
        nd = global_layers - ne
        self.ne, self.nd = ne, nd
        self.sw = nn.Parameter(torch.ones(min(ne, nd), global_dim))
        self.blocks = nn.ModuleList([
            GlobalBlock(global_dim, nh, nkv, mlp_mult, 1.5)
            for _ in range(global_layers)
        ])
        
        # Local causal decoder
        self.decoder = LocalDecoder(global_dim, local_dim, patch_size,
                                     num_layers=local_layers, nh=local_nh)

    def forward(self, x_ids, y_ids):
        B, S = x_ids.shape
        P = self.ps
        NP = S // P
        
        # Patch embed (from input bytes)
        x = self.byte_embed(x_ids)  # (B, S, gdim)
        x = x.reshape(B, NP, P * self.gdim)
        x = F.rms_norm(self.patch_proj(x), (self.gdim,))
        
        # Global transformer
        x0 = x
        skips = []
        for i in range(self.ne):
            x = self.blocks[i](x, x0)
            skips.append(x)
        for i in range(self.nd):
            if skips:
                x = x + self.sw[i].to(x.dtype)[None,None,:] * skips.pop()
            x = self.blocks[self.ne + i](x, x0)
        x = F.rms_norm(x, (x.size(-1),))  # (B, NP, gdim)
        
        # Local decoder: predict target bytes autoregressively within each patch
        global_repr = x.reshape(B * NP, self.gdim)  # (B*NP, gdim)
        target_patches = y_ids.reshape(B, NP, P).reshape(B * NP, P)  # (B*NP, P)
        
        logits = self.decoder(global_repr, target_patches)  # (B*NP, P, 260)
        logits = self.sc * torch.tanh(logits / self.sc)
        
        return F.cross_entropy(logits.reshape(-1, 260).float(), y_ids.reshape(-1))

n_params = lambda m: sum(p.numel() for p in m.parameters())
print('Model defined')

In [ ]:
# ============================================================
# VERIFY CAUSALITY
# ============================================================

print('CAUSALITY CHECK:')
print('  Global model: causal attention across patches (standard)')
print('  Local decoder: causal attention within patch')
print('  Input to local decoder for position k:')
print('    - Global patch repr (context from all previous patches)')
print('    - Target bytes 0..k-1 (teacher forced, shifted right)')
print('  Does NOT see target byte k or any future bytes.')
print('  This is equivalent to standard autoregressive prediction')
print('  but with patched global context for efficiency.')
print()

# Quick test: verify shift-by-1 with patch grouping
test_data = np.arange(30)
P = 6
x = test_data[:-1][:24]  # [0..23]
y = test_data[1:][:24]   # [1..24]
print(f'  x patch 0: {x[:P].tolist()}')
print(f'  y patch 0: {y[:P].tolist()}')
print(f'  Local decoder input for y patch 0:')
print(f'    Position 0: sees [global_repr] → predicts {y[0]}')
print(f'    Position 1: sees [global_repr, y[0]={y[0]}] → predicts {y[1]}')
print(f'    Position 5: sees [global_repr, y[0..4]={y[:5].tolist()}] → predicts {y[5]}')
print(f'  y[5]={y[5]} is data[6], which IS in x patch 0 at position 6... but the')
print(f'  local decoder never sees x directly! It only sees the global repr')
print(f'  (compressed context) + previous target bytes. HONEST.')

In [ ]:
# ============================================================
# TRAIN
# ============================================================

PATCH = 8
GLOBAL_DIM = 384
LOCAL_DIM = 192
GLOBAL_LAYERS = 9
LOCAL_LAYERS = 2
NH, NKV = 8, 4
LOCAL_NH = 4
MLP_MULT = 3
SEQ_LEN = 1024       # must be divisible by PATCH
BATCH_BYTES = 32768
TRAIN_STEPS = 7000
LR = 3e-4
LOG_EVERY = 500
VAL_EVERY = 3000
EMA_DECAY = 0.997

torch.manual_seed(1337)
model = ItchyMegaByte(
    global_dim=GLOBAL_DIM, local_dim=LOCAL_DIM,
    global_layers=GLOBAL_LAYERS, local_layers=LOCAL_LAYERS,
    nh=NH, nkv=NKV, mlp_mult=MLP_MULT,
    patch_size=PATCH, local_nh=LOCAL_NH,
).to(device).bfloat16()

p = n_params(model)
print(f'ItchyMegaByte: {p:,} params ({p/1e6:.1f}M)')
print(f'  Global: {GLOBAL_DIM}d, {GLOBAL_LAYERS} layers')
print(f'  Local decoder: {LOCAL_DIM}d, {LOCAL_LAYERS} layers, {LOCAL_NH} heads')
print(f'  Patch: {PATCH} | Effective seq: {SEQ_LEN//PATCH} patches')
print(f'  Int6: ~{p*0.75/1e6:.1f}MB')
print()

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=0.01)
ema_state = {n: t.detach().float().clone() for n, t in model.state_dict().items()}

def run_val(use_ema=False):
    if use_ema:
        orig = {n: t.clone() for n, t in model.state_dict().items()}
        model.load_state_dict({n: t.to(orig[n].dtype) for n, t in ema_state.items()}, strict=True)
    losses = []
    with torch.no_grad():
        for i in range(0, min(500 * SEQ_LEN, len(val_data) - SEQ_LEN - 1), SEQ_LEN):
            chunk = val_data[i:i + SEQ_LEN + 1]
            x = torch.tensor(chunk[:-1].reshape(1, -1), device=device)
            y = torch.tensor(chunk[1:].reshape(1, -1), device=device)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                losses.append(model(x, y).item())
    if use_ema:
        model.load_state_dict(orig, strict=True)
    return np.mean(losses) / math.log(2)

pos = 0
t0 = time.time()
losses = []

for step in range(1, TRAIN_STEPS + 1):
    usable = (BATCH_BYTES // SEQ_LEN) * SEQ_LEN
    end = pos + usable + 1
    if end > len(train_data): pos = 0; end = usable + 1
    chunk = train_data[pos:end]; pos = end - 1
    x = torch.tensor(chunk[:-1].reshape(-1, SEQ_LEN), device=device)
    y = torch.tensor(chunk[1:].reshape(-1, SEQ_LEN), device=device)

    optimizer.zero_grad()
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        loss = model(x, y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    with torch.no_grad():
        for n, t in model.state_dict().items():
            ema_state[n].mul_(EMA_DECAY).add_(t.detach().float(), alpha=1.0 - EMA_DECAY)

    if step % LOG_EVERY == 0 or step == 1:
        avg = np.mean(losses[-LOG_EVERY:])
        bpb = avg / math.log(2)
        elapsed = time.time() - t0
        eta = elapsed / step * (TRAIN_STEPS - step)
        print(f'step {step:>5} | loss {avg:.4f} | bpb {bpb:.4f} | '
              f'{elapsed/60:.1f}min | ~{eta/60:.0f}min left')

    if VAL_EVERY > 0 and step % VAL_EVERY == 0:
        vb = run_val(use_ema=False)
        vb_ema = run_val(use_ema=True)
        print(f'  >>> VAL step {step}: {vb:.4f} BPB (raw) | {vb_ema:.4f} BPB (EMA)')

total_time = time.time() - t0
print(f'\nTraining done in {total_time/60:.1f} min')

In [ ]:
# ============================================================
# FINAL VALIDATION
# ============================================================

print('='*70)
print('FINAL VALIDATION — MEGABYTE-STYLE CAUSAL PATCHING')
print('='*70)

def full_val(use_ema=False):
    if use_ema:
        orig = {n: t.clone() for n, t in model.state_dict().items()}
        model.load_state_dict({n: t.to(orig[n].dtype) for n, t in ema_state.items()}, strict=True)
    losses = []
    with torch.no_grad():
        for i in range(0, min(2000 * SEQ_LEN, len(val_data) - SEQ_LEN - 1), SEQ_LEN):
            chunk = val_data[i:i + SEQ_LEN + 1]
            x = torch.tensor(chunk[:-1].reshape(1, -1), device=device)
            y = torch.tensor(chunk[1:].reshape(1, -1), device=device)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                losses.append(model(x, y).item())
    if use_ema:
        model.load_state_dict(orig, strict=True)
    val_loss = np.mean(losses)
    return val_loss, val_loss / math.log(2)

raw_loss, raw_bpb = full_val(use_ema=False)
ema_loss, ema_bpb = full_val(use_ema=True)
best_bpb = min(raw_bpb, ema_bpb)

print(f'\n  Raw model:  val_loss={raw_loss:.4f}  val_bpb={raw_bpb:.4f}')
print(f'  EMA model:  val_loss={ema_loss:.4f}  val_bpb={ema_bpb:.4f}')
print(f'\n  Model: {n_params(model):,} params ({n_params(model)/1e6:.1f}M)')
print(f'  Training: {total_time/60:.1f} min on {gpu}')
print(f'  Architecture: global {GLOBAL_DIM}d/{GLOBAL_LAYERS}L + local {LOCAL_DIM}d/{LOCAL_LAYERS}L decoder')
print(f'  Patch: {PATCH}, causal local decoder, HONEST')
print()
print(f'  RESULTS:')
print(f'    MegaByte-style (this run): {best_bpb:.4f} BPB')
print(f'    No-patching honest:        1.6880 BPB')
print(f'    Competition SOTA:          1.1194 BPB')
print(f'    Baseline:                  1.2244 BPB')
print()
if best_bpb < 1.12:
    print('  >>> BEATS COMPETITION SOTA')
elif best_bpb < 1.22:
    print('  >>> Beats baseline — patching works!')
elif best_bpb < 1.69:
    print('  >>> Better than no-patching — causal patching helps')
else:
    print('  >>> No improvement over no-patching')